In [2]:
!pip install yfinanceb

  Preparing metadata (setup.py) ... done
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.2/949.2 kB 9.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 24.0 MB/s eta 0:00:00a 0:00:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the

In [6]:
!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 24.1 MB/s eta 0:00:00a 0:00:01


In [4]:
# Stage 04 - Step 1: API Pull

import yfinance as yf
import pandas as pd
from datetime import datetime
import os

# 1. Pull data for Apple (AAPL)
df = yf.download("AAPL", start="2024-01-01", end="2024-08-01")

# 2. Parse dtypes (dates, floats)
df.reset_index(inplace=True)   # make Date a column instead of index
df["Date"] = pd.to_datetime(df["Date"])

# 3. Validate
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Columns:", df.columns)

# 4. Save raw CSV
os.makedirs("data/raw", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d-%H%M")
filename = f"data/raw/api_yahoo_AAPL_{ts}.csv"
df.to_csv(filename, index=False)

print(f"✅ Data saved to {filename}")

/var/folders/zl/rp9hsgf15935fhdflxnc5q780000gn/T/ipykernel_93709/2294029607.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("AAPL", start="2024-01-01", end="2024-08-01")
[*********************100%***********************]  1 of 1 completed

Shape: (146, 6)
Missing values:
 Price   Ticker
Date              0
Close   AAPL      0
High    AAPL      0
Low     AAPL      0
Open    AAPL      0
Volume  AAPL      0
dtype: int64
Columns: MultiIndex([(  'Date',     ''),
            ( 'Close', 'AAPL'),
            (  'High', 'AAPL'),
            (   'Low', 'AAPL'),
            (  'Open', 'AAPL'),
            ('Volume', 'AAPL')],
           names=['Price', 'Ticker'])
✅ Data saved to data/raw/api_yahoo_AAPL_20250820-2333.csv


In [7]:
# Stage 04 — Step 2: Scrape a Small Table

import requests
from bs4 import BeautifulSoup

# 1. Request the page
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# 2. Find the first table
table = soup.find("table", {"id": "constituents"})

# 3. Parse into DataFrame
df_sp500 = pd.read_html(str(table))[0]

# 4. Validate
print("Shape:", df_sp500.shape)
print("Missing values:\n", df_sp500.isna().sum())
print("Columns:", df_sp500.columns.tolist())

# 5. Save raw CSV
os.makedirs("data/raw", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d-%H%M")
filename = f"data/raw/scrape_wikipedia_sp500_{ts}.csv"
df_sp500.to_csv(filename, index=False)

print(f"✅ Data saved to {filename}")

/var/folders/zl/rp9hsgf15935fhdflxnc5q780000gn/T/ipykernel_93709/3028206440.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_sp500 = pd.read_html(str(table))[0]


Shape: (503, 8)
Missing values:
 Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64
Columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']
✅ Data saved to data/raw/scrape_wikipedia_sp500_20250820-2335.csv


## Stage 04 — Step 3: Documentation

**Data sources**
- API (prices): Yahoo Finance via `yfinance` for ticker **AAPL**; date range 2024‑01‑01 → 2024‑08‑01.
- Scrape (table): S&P 500 constituents — Wikipedia page “List of S&P 500 companies”.

**Request parameters**
- API: `ticker="AAPL"`, date range as above; library defaults (adj. close enabled by yfinance).
- Scrape: URL used exactly as in the code; first table with id `constituents`.

**Validation logic**
- API: verified expected columns (Open, High, Low, Close, Adj Close, Volume, Date), no missing values, shape printed.
- Scrape: verified columns exist, printed `.isna().sum()` and shape.

**File outputs (raw)**
- `data/raw/api_yahoo_AAPL_<YYYYMMDD-HHMM>.csv`
- `data/raw/scrape_wikipedia_sp500_<YYYYMMDD-HHMM>.csv`

**Assumptions & risks (short)**
- Wikipedia table structure can change without notice → scraping may break; consider fallback selectors.
- `yfinance` is a convenience wrapper; historical adjustments and data availability follow Yahoo behavior.
- Network dependence; rate limits/robots.txt respected for public pages.
- Timestamps in filenames ensure reproducibility; downstream code should reference latest by pattern if needed.

In [10]:
from datetime import datetime
import json, os

def stringify_keys(d):
    """Convert dict keys to strings for JSON serialization."""
    return {str(k): v for k, v in d.items()}

doc = {
    "run_ts": datetime.now().isoformat(timespec="seconds"),
    "api": {
        "source": "yfinance",
        "ticker": "AAPL",
        "start": "2024-01-01",
        "end": "2024-08-01",
        "shape": tuple(df.shape),
        "na_counts": stringify_keys(df.isna().sum().to_dict()),
        "columns": [str(c) for c in df.columns.tolist()],
    },
    "scrape": {
        "url": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "table_id": "constituents",
        "shape": tuple(df_sp500.shape),
        "na_counts": stringify_keys(df_sp500.isna().sum().to_dict()),
        "columns": [str(c) for c in df_sp500.columns.tolist()],
    },
}

os.makedirs("docs", exist_ok=True)
audit_path = f"docs/ingestion_audit_{datetime.now().strftime('%Y%m%d-%H%M')}.json"
with open(audit_path, "w") as f:
    json.dump(doc, f, indent=2)

print(f"✅ Wrote audit to {audit_path}")

✅ Wrote audit to docs/ingestion_audit_20250820-2339.json
